In [ ]:
!pip install requests beautifulsoup4 lxml --quiet

In [ ]:
from google.colab import files

files.download('medlatec_diseases_crawl.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from tqdm import tqdm

import pandas as pd
import re

# Medlatec

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://medlatec.vn/tu-dien-benh-ly/benh-alzheimer-scjhe"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123 Safari/537.36"
}

r = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

sections = soup.find_all("section")
print("Số section:", len(sections))

for sec in sections:
    print(sec.get("id"))


Số section: 10
smooth-scroll-link
disease-description
disease-causes
disease-symptoms_free
disease-Complications
disease-ObjectsAtRisk
disease-Prevent
disease-diagnostic
disease-Treatment
None


In [ ]:
section_ids = ["disease-description", "disease-causes", "disease-symptoms_free",
              "disease-Complications", "disease-ObjectsAtRisk", "disease-Prevent",
              "disease-diagnostic", "disease-Treatment"]

section_names = ["Tổng quan", "Nguyên nhân", "Triệu chứng",
                 "Biến chứng", "Đối tượng nguy cơ", "Phòng ngừa",
                 "Chẩn đoán", "Điều trị"]

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0"}

In [ ]:
# Crawl page về bệnh
def crawl_a_page(url, disease_name):

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    results = []   # *** list chứa nội dung theo thứ tự section_names ***

    for sec_id in section_ids:
        section = soup.find("section", id=sec_id)

        if section:
            text = section.get_text("\n", strip=True)

            # bỏ dòng đầu tiên (tiêu đề)
            lines = text.split("\n")[1:]

            # xử lý riêng phần Điều trị (bỏ quảng cáo)
            if sec_id == "disease-Treatment":
                clean_lines = []
                for line in lines:
                    if "MEDLATEC" in line or "Chuyên khoa" in line:
                        break
                    clean_lines.append(line)
                lines = clean_lines

            clean_text = "\n".join(lines)

        else:
            clean_text = "No in4 found"   # nếu section không tồn tại trong trang

        results.append(clean_text)  # lưu vào list theo đúng thứ tự

    # Trả về DataFrame 1 dòng
    df = pd.DataFrame([results], columns=section_names)

    # Thêm TÊN BỆNH vào DataFrame
    df["Tên bệnh"] = disease_name
    return df

In [ ]:
BASE = "https://medlatec.vn"
URL_TEMPLATE = "https://medlatec.vn/tu-dien-benh-ly/?Alphabet={}&Search="

all_items = []   # Lưu (url, name)

# Lặp từ A → Z
for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":  # thử chỉ chữ A
    url = URL_TEMPLATE.format(c)
    print("Đang lấy trang:", url)

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if href.startswith("/tu-dien-benh-ly/") and len(href) > len("/tu-dien-benh-ly/"):
            full = BASE + href
            disease_name = a.get_text(strip=True)
            all_items.append((full, disease_name))

print("\nTổng số link bệnh lấy được:", len(all_items))

all_dfs = []

for link, d_name in all_items:
    print("Đang crawl:", d_name)
    try:
        df = crawl_a_page(link, d_name)   # TRUYỀN d_name
        df["URL"] = link
        all_dfs.append(df)
    except Exception as e:
        print("❌ Lỗi ở link:", link, "Error:", e)


In [ ]:
final_df = pd.concat(all_dfs, ignore_index=True)

print("\n=== DONE ===")
print("Shape:", final_df.shape)
display(final_df)


=== DONE ===
Shape: (1060, 10)


,Tổng quan,Nguyên nhân,Triệu chứng,Biến chứng,Đối tượng nguy cơ,Phòng ngừa,Chẩn đoán,Điều trị,Tên bệnh,URL
0,Áp xe là một túi mủ ở một vùng rỗng của cơ thể...,Áp xe thận có thể do vi khuẩn từ nhiễm trùng đ...,Các dấu hiệu và triệu chứng phổ biến ở bệnh nh...,Biến chứng đáng sợ nhất của áp xe nhu mô thận ...,No in4 found,No in4 found,Đôi khi có thể mất một thời gian để phát hiện ...,"Ở hầu hết bệnh nhân bị viêm thận bể thận cấp, ...",Áp xe thận,https://medlatec.vn/tu-dien-benh-ly/ap-xe-than...
1,Bệnh ấu trùng sán dây lợn (cysticercosis) do\n...,Sán dây lợn\nTeania solium\nthuộc giống sán dâ...,1. Triệu chứng lâm sàng\na. Thể bệnh ấu trùng ...,"Một số biến chứng về thần kinh như co giật, độ...",Bệnh ấu trùng sán dây lợn ghi nhận nhiều khu v...,"- Nâng cao nhận thức, hiểu biết, truyền thông,...",Chẩn đoán bệnh ấu trùng sán lợn hệ cần dựa vào...,Kiểm soát tăng áp lực nội sọ\n- Corticosteroid...,Ấu trùng sán dây lợn,https://medlatec.vn/tu-dien-benh-ly/au-trung-s...
2,"Áp xe vú là ổ chứa đầy mủ, xung quanh là mô vi...","Trong áp xe vú, hay gặp nhất là vi khuẩn tụ cầ...","Với áp xe vú sau sinh, có hai giai đoạn người ...",No in4 found,No in4 found,Bệnh áp xe vú thường xảy ra ở phụ nữ sau sinh ...,No in4 found,1. Nguyên tắc điều trị:\nHạn chế nhiễm trùng v...,Áp xe vú,https://medlatec.vn/tu-dien-benh-ly/ap-xe-vu-s...
3,- Áp Xe quanh Amydal là tình trạng viêm mủ cấp...,"- Thường đo viêm Amydal mạn tính đợt cấp, hoặc...","Giai đoạn khởi đầu: Xung huyết, viêm tấy đơn t...",Áp xe quanh Amydal\nlà bệnh lý có tính chất cấ...,"Bệnh thường hay gặp ở thanh thiếu niên, người ...","- Nâng cao mức sống, tăng cường sức đề kháng c...",Chẩn đoán xác định\n- Triệu chứng toàn thân: S...,- Trước khi túi mủ hình thành: Điều trị nội k...,Áp Xe quanh Amydal,https://medlatec.vn/tu-dien-benh-ly/ap-xe-quan...
4,Là một bộ phận vô cùng quan trọng trong hệ thố...,"Theo như chúng tôi đã đề cập, thì các yếu tố n...",Áp xe não có thể phát triển triệu chứng muộn t...,No in4 found,Áp xe não không loại trừ bất kỳ đối tượng nào ...,- Đối với những áp xe não có nguyên nhân xuất ...,Bên cạnh công tác đánh giá những triệu chứng (...,Với sự tiến bộ của khoa học hiện nay đã mở ra...,Áp xe não,https://medlatec.vn/tu-dien-benh-ly/ap-xe-nao-...
...,...,...,...,...,...,...,...,...,...,...
1055,Xơ cứng bì toàn thể là gì?\nXơ cứng bì toàn th...,Xơ cứng bì toàn thể là một bệnh lý tự miễn có ...,Xơ cứng bì toàn thể thường tiến triển âm thầm ...,Xơ cứng bì toàn thể là bệnh tự miễn mạn tính c...,Xơ cứng bì là một bệnh hiếm gặp. Tỷ lệ hiện mắ...,No in4 found,"Tiêu chuẩn chẩn đoán\nHiện nay, tiêu chuẩn phâ...","Hiện nay, xơ cứng bì toàn thể chưa có phương p...",Xơ cứng bì toàn thể,https://medlatec.vn/tu-dien-benh-ly/xo-cung-bi...
1056,Xuất huyết dưới màng nhện (Subarachnoid Hemorr...,Xuất huyết dưới màng nhện chủ yếu phát sinh do...,Triệu chứng khởi phát điển hình là:\nĐau đầu k...,Hậu quả nghiêm trọng của SAH bao gồm:\nTái xuấ...,"Tại Hoa Kỳ, SAH chiếm khoảng 3% các trường hợp...",No in4 found,Chẩn đoán sớm và chính xác xuất huyết dưới màn...,Xuất huyết dưới màng nhện là tình trạng cấp cứ...,Xuất huyết dưới màng nhện,https://medlatec.vn/tu-dien-benh-ly/xuat-huyet...
1057,Xuất huyết võng mạc là gì?\nXuất huyết võng mạ...,Xuất huyết võng mạc không phải là một bệnh lý ...,"Xuất huyết võng mạc có thể diễn biến âm thầm, ...",Tiên lượng của bệnh xuất huyết võng mạc phụ th...,No in4 found,No in4 found,Tiêu chuẩn chẩn đoán\nViệc xác định xuất huyết...,Điều trị xuất huyết võng mạc không chỉ tập tru...,Xuất huyết võng mạc,https://medlatec.vn/tu-dien-benh-ly/xuat-huyet...
1058,Xơ vữa động mạch não là gì?\nXơ vữa động mạch ...,Xơ vữa động mạch não hình thành từ một quá trì...,No in4 found,No in4 found,No in4 found,Xơ vữa động mạch não là bệnh mạn tính nhưng ho...,Chẩn đoán xơ vữa động mạch não đóng vai trò qu...,Mục tiêu điều trị xơ vữa động mạch não là phòn...,Xơ vữa động mạch não,https://medlatec.vn/tu-dien-benh-ly/xo-vua-don...


In [ ]:
final_df.to_csv("medlatec_diseases_crawl_final.csv", index=False, encoding="utf-8-sig")


In [ ]:
BASE = "https://medlatec.vn"
URL_TEMPLATE = "https://medlatec.vn/tu-dien-benh-ly/?Alphabet={}&Search="

all_items = []   # lưu (url, name)

# Lặp từ A → Z
for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
    url = URL_TEMPLATE.format(c)
    print("Đang lấy trang:", url)

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # lấy tất cả thẻ <a>
    for a in soup.find_all("a", href=True):
        href = a["href"]

        # chỉ lấy link dạng /tu-dien-benh-ly/<slug>
        if href.startswith("/tu-dien-benh-ly/") and len(href) > len("/tu-dien-benh-ly/"):
            full_url = BASE + href
            disease_name = a.get_text(strip=True)
            print(disease_name)
            all_items.append((full_url, disease_name))

print("\nTổng số link bệnh lấy được:", len(all_items))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#VinMec

In [ ]:
BASE = "https://www.vinmec.com/vie/benh"
URL_TEMPLATE = "https://www.vinmec.com/vie/tra-cuu-benh/{}"

all_links = set()

# Lặp từ A → Z
for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
    url = URL_TEMPLATE.format(c)
    print("Đang lấy trang:", url)

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # lấy tất cả thẻ <a>
    for a in soup.find_all("a", href=True):
        href = a["href"]

        # chỉ lấy link dạng /tu-dien-benh-ly/<slug>
        if href.startswith("/vie/benh/") and len(href) > len("/vie/benh/"):
            full = BASE + href
            all_links.add(full)

print("\nTổng số link bệnh lấy được:", len(all_links))

Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/A
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/B
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/C
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/D
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/E
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/F
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/G
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/H
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/I
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/J
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/K
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/L
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/M
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/N
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/O
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/P
Đang lấy trang: https://www.vinmec.com/vie/tra-cuu-benh/Q
Đang lấy trang

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.vinmec.com"
URL_TEMPLATE = "https://www.vinmec.com/vie/tra-cuu-benh/{}"

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_links_set = set()     # dùng để tránh trùng
all_links_list = []       # đây là ARRAY bạn yêu cầu
count_per_letter = {}     # số lượng theo từng chữ cái

def get_links_from_page(url):
    """Lấy link bài bệnh từ một trang tra cứu bệnh."""
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        return set()

    soup = BeautifulSoup(r.text, "html.parser")
    page_links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if href.startswith("/vie/benh/") and len(href) > len("/vie/benh/"):
            full = urljoin(BASE, href)
            page_links.add(full)

    return page_links


# ============================================
# Lặp A → Z
# ============================================

for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
    print(f"\n🔎 Đang lấy danh sách theo chữ: {c}")

    page = 0
    collected = set()

    while True:
        url = URL_TEMPLATE.format(c) if page == 0 else URL_TEMPLATE.format(c) + f"/page_{page}"

        print("   → Crawling:", url)
        links = get_links_from_page(url)

        if not links:
            print("     ⛔ Không còn trang → Dừng chữ", c)
            break

        print(f"     ✔ Lấy được {len(links)} link")

        for link in links:
            if link not in all_links_set:
                all_links_set.add(link)
                all_links_list.append(link)   # 👉 thêm vào ARRAY

        collected.update(links)
        page += 1

    count_per_letter[c] = len(collected)
    print(f"📌 Tổng link chữ {c}: {len(collected)}")


# ============================================
# In kết quả
# ============================================

print("\n==============================")
print("📦 Tổng số link bệnh lấy được:", len(all_links_list))
print("==============================")

# In thử 10 link đầu tiên
for link in all_links_list[:10]:
    print(link)



🔎 Đang lấy danh sách theo chữ: A
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/A
     ✔ Lấy được 17 link
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/A/page_1
     ✔ Lấy được 17 link
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/A/page_2
     ⛔ Không còn trang → Dừng chữ A
📌 Tổng link chữ A: 17

🔎 Đang lấy danh sách theo chữ: B
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/B
     ✔ Lấy được 30 link
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/B/page_1
     ✔ Lấy được 30 link
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/B/page_2
     ✔ Lấy được 7 link
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/B/page_3
     ⛔ Không còn trang → Dừng chữ B
📌 Tổng link chữ B: 37

🔎 Đang lấy danh sách theo chữ: C
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/C
     ✔ Lấy được 28 link
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/C/page_1
     ✔ Lấy được 28 link
   → Crawling: https://www.vinmec.com/vie/tra-cuu-benh/C/pag

In [ ]:
print(len(all_links_list))


676


In [ ]:
def crawl_a_page(url):
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # ---------------------------------------------------
    # 1. Lấy tên bệnh
    # ---------------------------------------------------
    name_tag = soup.find("div", class_="f30 bold mb2")
    ten_benh_raw = name_tag.get_text(strip=True) if name_tag else ""

    ten_benh = ten_benh_raw.split(":")[0]
    ten_benh = ten_benh.split("(")[0].strip()

    # ---------------------------------------------------
    # 2. Khởi tạo kết quả rỗng
    # ---------------------------------------------------
    result = {
        "Tên bệnh": ten_benh,
        "Tổng quan": "",
        "Nguyên nhân": "",
        "Triệu chứng": "",
        "Đường lây truyền": "",
        "Đối tượng": "",
        "Phòng ngừa": "",
        "Biện pháp chẩn đoán": "",
        "Các biện pháp điều trị": ""
    }

    # ---------------------------------------------------
    # 3. Mapping theo từ khóa xuất hiện trong tiêu đề
    # ---------------------------------------------------
    keyword_map = {
        "tổng quan": "Tổng quan",
        "nguyên nhân": "Nguyên nhân",
        "triệu chứng": "Triệu chứng",
        "lây truyền": "Đường lây truyền",
        "đối tượng": "Đối tượng",
        "phòng ngừa": "Phòng ngừa",
        "chẩn đoán": "Biện pháp chẩn đoán",
        "điều trị": "Các biện pháp điều trị"
    }

    # ---------------------------------------------------
    # 4. Lấy tất cả cặp <h2> + <div class="body collapsible-target">
    # ---------------------------------------------------
    titles = soup.find_all("h2", class_="title_detail_sick")
    bodies = soup.find_all("div", class_="body collapsible-target")

    # ghép đúng theo cặp (Vinmec luôn <h2> → <div>)
    for title, body in zip(titles, bodies):

        title_text = title.get_text(strip=True).lower()   # chuẩn hóa chữ thường
        body_text = body.get_text("\n", strip=True)

        # Tìm cột nào phù hợp
        matched_column = None
        for kw, col in keyword_map.items():
            if kw in title_text:
                matched_column = col
                break

        if not matched_column:
            continue  # không match bất kỳ cột nào → bỏ qua

        # Xử lý riêng phần điều trị (loại "Xem thêm:")
        if matched_column == "Các biện pháp điều trị":
            body_text = body_text.split("Xem thêm:")[0].strip()

        # Lưu vào đúng cột
        result[matched_column] = body_text

    # Trả về DataFrame
    df = pd.DataFrame([result])
    return df


In [ ]:
data = crawl_a_page("https://www.vinmec.com/vie/benh/addison-suy-tuyen-thuong-than-nguyen-phat-4696")
display(data)

In [ ]:
all_dfs = []

for i, link in enumerate(all_links_list, 1):
    print(f"🔎 Đang crawl {i}/{len(all_links_list)}:", link)
    try:
        df = crawl_a_page(link)
        all_dfs.append(df)
    except Exception as e:
        print("❌ Lỗi tại:", link, "| Error:", e)

# MERGE toàn bộ
final_df = pd.concat(all_dfs, ignore_index=True)

print("\n🎉 Crawl xong! Tổng số bệnh:", len(final_df))
final_df.head()


In [ ]:
urls = ["https://www.vinmec.com/vie/benh/ap-xe-gan-3238", "https://www.vinmec.com/vie/benh/addison-suy-tuyen-thuong-than-nguyen-phat-4696"]

all_dfs = []
for i in urls:
  df = crawl_a_page(i)
  all_dfs.append(df)

final_df = pd.concat(all_dfs, ignore_index=True)
final_df.head()

In [ ]:
final_df.to_csv("vinmec_all_diseases.csv", index=False, encoding="utf-8-sig")
print("Đã lưu CSV → vinmec_all_diseases_fin.csv")

Đã lưu CSV → vinmec_all_diseases_fin.csv


# Youmed

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_disease_list():
    url = "https://youmed.vn/tin-tuc/trieu-chung-benh/"
    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    results = []

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        name = a.get_text(strip=True)

        # Chỉ lấy link bài bệnh
        if href.startswith("https://youmed.vn/tin-tuc/") and href.count("/") > 4:
            if len(name) > 1:  # tránh lấy chữ cái A, B, C...
                results.append({
                    "name": name,
                    "link": href
                })

    return results


# ====== Test ======
diseases = get_disease_list()
for d in diseases:
  print(d)
print("Tổng số:", len(diseases))


{'name': 'Bệnh bạch hầu', 'link': 'https://youmed.vn/tin-tuc/benh-bach-hau-nhan-biet-som-va-phong-tranh/'}
{'name': 'Rối loạn tiền đình', 'link': 'https://youmed.vn/tin-tuc/roi-loan-tien-dinh-va-cach-phong-tranh/'}
{'name': 'Sốt xuất huyết', 'link': 'https://youmed.vn/tin-tuc/sot-xuat-huyet-trieu-chung-cach-dieu-tri-va-nhung-luu-y/'}
{'name': 'Trào ngược dạ dày thực quản', 'link': 'https://youmed.vn/tin-tuc/trao-nguoc-da-day-thuc-quan-dau-hieu-va-bien-phap-ho-tro/'}
{'name': 'Viêm da dị ứng', 'link': 'https://youmed.vn/tin-tuc/viem-da-di-ung-nguyen-nhan-trieu-chung-chuan-doan/'}
{'name': 'Đột quỵ', 'link': 'https://youmed.vn/tin-tuc/ban-biet-gi-ve-phuc-hoi-chuc-nang-sau-dot-quy/'}
{'name': 'Bệnh Parkinson', 'link': 'https://youmed.vn/tin-tuc/benh-parkinson-co-trieu-chung-nhu-the-nao/'}
{'name': 'Tra cứu thêm', 'link': 'https://youmed.vn/tin-tuc/trieu-chung-benh/'}
{'name': 'Thuốc & Thực phẩm chức năng', 'link': 'https://youmed.vn/tin-tuc/duoc/'}
{'name': 'Paracetamol', 'link': 'https:/

In [ ]:
# ----- 1. Bảng từ khóa map vào các cột -----
SECTION_MAP = {
    "Tổng quan": ["tổng quan", "giới thiệu", "là gì"],
    "Nguyên nhân": ["nguyên nhân", "nguyên do"],
    "Dấu hiệu": ["dấu hiệu", "triệu chứng", "nhận biết"],
    "Biến chứng": ["biến chứng"],
    "Chẩn đoán": ["chẩn đoán", "xét nghiệm"],
    "Điều trị": ["điều trị", "chữa", "phương pháp"],
}

# ----- 2. Hàm nhận diện section thuộc cột nào -----
def map_section(title):
    title_lower = title.lower()

    for col, keywords in SECTION_MAP.items():
        for kw in keywords:
            if kw in title_lower:
                return col
    return None   # nếu không khớp thì bỏ qua


# ----- 3. Crawl lấy H2 + nội dung -----
def crawl_page(url, ten_benh):
    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    data = {
        "Tên bệnh": ten_benh,
        "Tổng quan": "",
        "Nguyên nhân": "",
        "Dấu hiệu": "",
        "Biến chứng": "",
        "Chẩn đoán": "",
        "Điều trị": ""
    }

    # lấy tất cả h2
    h2_tags = soup.find_all("h2")

    for h2 in h2_tags:
        title = h2.get_text(strip=True)
        col = map_section(title)
        if not col:
            continue  # bỏ section không thuộc nhóm cần

        # Lấy nội dung phía sau h2 đến h2 tiếp theo
        content_parts = []
        sibling = h2.find_next_sibling()

        while sibling and sibling.name != "h2":
            if sibling.name in ["p", "div", "ul", "ol"]:
                text = sibling.get_text(" ", strip=True)
                if text:
                    content_parts.append(text)
            sibling = sibling.find_next_sibling()

        data[col] = "\n".join(content_parts)

    return data

data = crawl_page('https://youmed.vn/tin-tuc/benh-bach-hau-nhan-biet-som-va-phong-tranh/', "test")


In [ ]:
data

{'Tên bệnh': 'test',
 'Tổng quan': 'Thời kì ủ bệnh thường kéo dài 2 – 5 ngày. Biểu hiện bệnh được phân chia dựa vào vị trí:\n1. Mũi trước\nThường khởi phát giống cảm lạnh. Đặc trưng bởi chảy mũi nhày mủ, có thể lẫn máu. Màng giả máu trắng xám thường được tạo thành ở vách ngăn. Bệnh thường nhẹ, do sự hấp thu độc tố vào máu tại chỗ kém. Bệnh có thể điều trị được bằng kháng độc tố và kháng sinh.\n2. Họng và amidan\nVị trí thường gặp nhất của bạch hầu. Giai đoạn sớm thường mệt mỏi, đau họng, chán ăn, sốt nhẹ. Trong vòng 2 – 3 ngày, hình thành một màng màu trắng xanh, kích thước rất thay đổi. Có thể nhỏ như một mảnh vá trên bề mặt amidan, có thể lớn che phủ gần hết vùng họng. Một số bệnh nhân có thể tự lui bệnh mà không cần điều trị.\nMột số khác có thể tiến triển nặng. Sốt thường không cao, ngay cả khi nhiễm độc. Bệnh nhân nặng có thể sưng to vùng dưới hàm, hạch cổ. Nếu độc tố đi vào máu nhiều, người bệnh sẽ phờ phạc, tím tái, mạch nhanh, lờ đờ, hôn mê. Có thể tử vong trong vòng 6 đến 10 n

In [ ]:
link_list = get_disease_list()
rows = []

print("Tổng số link tìm được:", len(link_list))

for item in link_list:
    ten = item["name"]
    link = item["link"]

    print(f"Crawl: {ten} | {link}")

    try:
        row = crawl_page(link, ten)
        rows.append(row)
    except Exception as e:
        print(f"Lỗi khi crawl {ten}: {e}")

# -------------------
# 5. Xuất DataFrame thành CSV
# -------------------

df = pd.DataFrame(rows)
df.to_csv("youmed_diseases_full.csv", index=False, encoding="utf-8")

print("\n DONE! Đã lưu file youmed_diseases_full.csv")
df.head()

Tổng số link tìm được: 713
🔍 Crawl: Bệnh bạch hầu | https://youmed.vn/tin-tuc/benh-bach-hau-nhan-biet-som-va-phong-tranh/
🔍 Crawl: Rối loạn tiền đình | https://youmed.vn/tin-tuc/roi-loan-tien-dinh-va-cach-phong-tranh/
🔍 Crawl: Sốt xuất huyết | https://youmed.vn/tin-tuc/sot-xuat-huyet-trieu-chung-cach-dieu-tri-va-nhung-luu-y/
🔍 Crawl: Trào ngược dạ dày thực quản | https://youmed.vn/tin-tuc/trao-nguoc-da-day-thuc-quan-dau-hieu-va-bien-phap-ho-tro/
🔍 Crawl: Viêm da dị ứng | https://youmed.vn/tin-tuc/viem-da-di-ung-nguyen-nhan-trieu-chung-chuan-doan/
🔍 Crawl: Đột quỵ | https://youmed.vn/tin-tuc/ban-biet-gi-ve-phuc-hoi-chuc-nang-sau-dot-quy/
🔍 Crawl: Bệnh Parkinson | https://youmed.vn/tin-tuc/benh-parkinson-co-trieu-chung-nhu-the-nao/
🔍 Crawl: Tra cứu thêm | https://youmed.vn/tin-tuc/trieu-chung-benh/
🔍 Crawl: Thuốc & Thực phẩm chức năng | https://youmed.vn/tin-tuc/duoc/
🔍 Crawl: Paracetamol | https://youmed.vn/tin-tuc/paracetamol-acetaminophen-cong-dung-va-cach-dung/
🔍 Crawl: Glucosamine |

,Tên bệnh,Tổng quan,Nguyên nhân,Dấu hiệu,Biến chứng,Chẩn đoán,Điều trị
0,Bệnh bạch hầu,Thời kì ủ bệnh thường kéo dài 2 – 5 ngày. Biểu...,Vi khuẩn Corynebacterium diphtheriae là tác nh...,,Hầu hết biến chứng của bạch hầu là do độc tố g...,,1. Kháng độc tố (Diphtheria Antitoxin)\nĐược s...
1,Rối loạn tiền đình,Tiền đình là cơ quan nằm ở phía sau ốc tai ở h...,Điều trị CMTTKPLT bao gồm tái định vị sỏi ống ...,Biểu hiện chính của rối loạn tiền đình là chón...,,Tình trạng bệnh có thể được chẩn đoán ban đầu ...,"Các biểu hiện chóng mặt, mất thăng bằng, ù tai..."
2,Sốt xuất huyết,Sốt xuất huyết là bệnh truyền nhiễm cấp tính d...,,Đây là dạng có biểu hiện các triệu chứng điển ...,,Bác sĩ có thể sẽ hỏi về lịch sử y tế và du lịc...,Sốt xuất huyết chưa có thuốc điều trị đặc hiệu...
3,Trào ngược dạ dày thực quản,Trào ngược dạ dày – thực quản còn được gọi là ...,Có thể kể đến hai nguyên nhân điển hình gây ra...,"Ợ hơi, ợ chua, ợ nóng: ợ hơi lúc đói thường xu...",,,Thay đổi lối sống có thể làm giảm tần suất trà...
4,Viêm da dị ứng,Đây là phản ứng của cơ thể đối với tác nhân bê...,Nguyên nhân gây bệnh là do sự tương tác qua lạ...,Biểu hiện của viêm da dị ứng tùy thuộc vào ngu...,,Có thể chẩn đoán viêm da dị ứng dựa vào yếu tố...,Việc điều trị viêm da dị ứng tùy thuộc vào cơ ...


In [ ]:
df.to_csv("youmed_diseases_full_fixed.csv", index=False, encoding="utf-8-sig")
